In [4]:
from pymongo import MongoClient
from pprint import pprint

URI_REPLICA = (
    "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/"
    "?replicaSet=my-replica-set&readPreference=primary&ssl=false"
)

client = MongoClient(URI_REPLICA)
db = client["salud_usm"]

print("Conexión establecida con el clúster:", client.topology_description)
print("Colecciones disponibles:", db.list_collection_names())

Conexión establecida con el clúster: <TopologyDescription id: 6ae426e28e214cce766c110f, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 30001) server_type: Unknown, rtt: None>, <ServerDescription ('mongo2', 30002) server_type: Unknown, rtt: None>, <ServerDescription ('mongo3', 30003) server_type: Unknown, rtt: None>]>
Colecciones disponibles: ['medicos', 'especialidades', 'centros_medicos', 'pacientes', 'previsiones', 'citas']


### 4.3.a Agenda diaria de un médico

**Requerimiento:**
Dado el RUT de un médico y una fecha, devolver sus citas de ese día ordenadas por hora, mostrando hora, nombre del paciente, previsión, centro, especialidad y estado.

**Parámetros de prueba:**
* **RUT Médico:** `'22222222-2'` (Dr. Juan Pérez)
* **Fecha:** `'2026-10-15'`

**Explicación de la solución MQL:**
1. `$match`: Filtra los documentos de la colección `citas` que coincidan exactamente con el `medico.rut` y la `fecha` especificados.
2. `$sort`: Ordena cronológicamente los resultados de manera ascendente (`1`) según el campo `hora`.
3. `$project`: Proyecta únicamente los atributos requeridos (`hora`, `especialidad`, `estado`) y extrae los campos embebidos (`paciente.nombre`, `paciente.prevision`, `centro.nombre`), suprimiendo el identificador por defecto `_id: 0`.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT c.hora,
       p.nombre_completo AS paciente,
       p.nombre_prevision AS prevision,
       cm.nombre_centro AS centro,
       c.nombre_especialidad AS especialidad,
       c.estado
FROM cita_medica c
JOIN paciente p ON c.rut_paciente = p.rut_paciente
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
WHERE c.rut_medico = '22222222-2'
  AND c.fecha = '2026-10-15'
ORDER BY c.hora ASC;

In [11]:
pipeline_a = [
    {
        "$match": {
            "medico.rut": "22222222-2",
            "fecha": "2026-10-15"
        }
    },
    {
        "$sort": { "hora": 1 }
    },
    {
        "$project": {
            "_id": 0,
            "hora": 1,
            "paciente": "$paciente.nombre",
            "prevision": "$paciente.prevision",
            "centro": "$centro.nombre",
            "especialidad": 1,
            "estado": 1
        }
    }
]

res_a = list(db.citas.aggregate(pipeline_a))
pprint(res_a)

[{'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Reservada',
  'hora': '09:00:00',
  'paciente': 'Juan Silva',
  'prevision': 'Fonasa'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Confirmada',
  'hora': '09:30:00',
  'paciente': 'Camila Soto',
  'prevision': 'Fonasa'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Cardiología',
  'estado': 'Confirmada',
  'hora': '10:00:00',
  'paciente': 'Pedro Morales',
  'prevision': 'Isapre'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Medicina General',
  'estado': 'Reservada',
  'hora': '10:30:00',
  'paciente': 'Francisca Araya',
  'prevision': 'Particular'},
 {'centro': 'Centro Médico Casa Central',
  'especialidad': 'Cardiología',
  'estado': 'Confirmada',
  'hora': '11:00:00',
  'paciente': 'Andrés Castro',
  'prevision': 'Fonasa'}]


### 4.3.b Historial clínico de un paciente

**Requerimiento:**
Dado el RUT de un paciente, devolver todas sus atenciones ordenadas de la más reciente a la más antigua, mostrando fecha, centro, especialidad, nombre del médico, motivo de consulta, diagnósticos (código CIE-10 y descripción) y recetas (medicamento, dosis y días de tratamiento).

**Parámetros de prueba:**
* **RUT Paciente:** `'33333333-3'` (Juan Silva)

**Explicación de la solución MQL:**
1. `$match`: Selecciona únicamente las citas del paciente cuyo estado sea `'Atendida'` y que contengan el documento embebido `atencion` (`{ "$exists": True }`).
2. `$sort`: Aplica ordenamiento descendente (`-1`) sobre `fecha` y `hora` para presentar las consultas clínicas desde la más reciente a la más antigua.
3. `$project`: Reestructura la salida omitiendo `_id`, extrayendo la metadata del centro, especialidad y médico, y proyectando directamente los arreglos anidados `diagnosticos` y `recetas` sin requerir descomposiciones ni uniones costosas.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT c.fecha,
       cm.nombre_centro AS centro,
       c.nombre_especialidad AS especialidad,
       m.nombre_completo AS medico,
       a.motivo_consulta,
       d.codigo_cie10,
       d.descripcion,
       r.medicamento,
       r.dosis,
       r.dias_tratamiento
FROM atencion a
JOIN cita_medica c ON a.id_cita = c.id_cita
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
JOIN medico m ON c.rut_medico = m.rut_medico
LEFT JOIN atencion_diagnostico ad ON a.id_atencion = ad.id_atencion
LEFT JOIN diagnostico d ON ad.codigo_cie10 = d.codigo_cie10
LEFT JOIN receta r ON a.id_atencion = r.id_atencion
WHERE c.rut_paciente = '33333333-3'
  AND c.estado = 'Atendida'
ORDER BY c.fecha DESC, c.hora DESC;

In [6]:
pipeline_b = [
    {
        "$match": {
            "paciente.rut": "33333333-3",
            "estado": "Atendida",
            "atencion": { "$exists": True }
        }
    },
    {
        "$sort": { "fecha": -1, "hora": -1 }
    },
    {
        "$project": {
            "_id": 0,
            "fecha": 1,
            "centro": "$centro.nombre",
            "especialidad": 1,
            "medico": "$medico.nombre",
            "motivo_consulta": "$atencion.motivo_consulta",
            "diagnosticos": "$atencion.diagnosticos",
            "recetas": "$atencion.recetas"
        }
    }
]

res_b = list(db.citas.aggregate(pipeline_b))
pprint(res_b)

[{'centro': 'Centro Médico Viña del Mar',
  'diagnosticos': [{'codigo_cie10': 'M54',
                    'descripcion': 'Dorsalgia / Lumbalgia'}],
  'especialidad': 'Traumatología',
  'fecha': '2026-09-30',
  'medico': 'Dr. Jorge Valenzuela',
  'motivo_consulta': 'Dolor cervical agudo',
  'recetas': [{'dias_tratamiento': 7,
               'dosis': 'Aplicar masaje suave cada 8 horas',
               'id_receta': 72,
               'medicamento': 'Ketoprofeno Gel 2.5%'}]},
 {'centro': 'Centro Médico Casa Central',
  'diagnosticos': [{'codigo_cie10': 'H10',
                    'descripcion': 'Conjuntivitis aguda'}],
  'especialidad': 'Oftalmología',
  'fecha': '2026-09-04',
  'medico': 'Dra. Valentina Castro',
  'motivo_consulta': 'Ardor ocular y enrojecimiento',
  'recetas': [{'dias_tratamiento': 7,
               'dosis': '1 gota cada 6 horas en ojo afectado',
               'id_receta': 32,
               'medicamento': 'Ciprofloxacino Oftálmico'}]},
 {'centro': 'Centro Médico Casa Cen

### 4.3.c Tasa de inasistencia por centro

**Requerimiento:**
Para cada centro médico, mostrar el total de citas, la cantidad de citas en estado 'No Asistió' y el porcentaje que representan (redondeado a 2 decimales), listando únicamente los centros cuyo porcentaje sea mayor a 15% y ordenando de mayor a menor porcentaje.

**Explicación de la solución MQL:**
1. `$group`: Agrupa por `centro.nombre`, calculando `total_citas` mediante un acumulador `$sum: 1` y `no_asistio` mediante un `$sum` condicional con `$cond` que evalúa si el estado es `'No Asistió'`.
2. `$project`: Calcula el porcentaje dividiendo citas no asistidas por el total, multiplicando por 100 y aplicando la función matemática `$round` a 2 posiciones decimales.
3. `$match`: Filtra únicamente los documentos agrupados cuyo `porcentaje_inasistencia` supere el umbral estipulado (`> 15.0`).
4. `$sort`: Ordena los centros resultantes de forma descendente (`-1`) por su tasa de inasistencia.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT cm.nombre_centro AS centro,
       COUNT(*) AS total_citas,
       SUM(CASE WHEN c.estado = 'No Asistió' THEN 1 ELSE 0 END) AS no_asistio,
       ROUND((SUM(CASE WHEN c.estado = 'No Asistió' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS porcentaje_inasistencia
FROM cita_medica c
JOIN centro_medico cm ON c.codigo_centro = cm.codigo_centro
GROUP BY cm.codigo_centro, cm.nombre_centro
HAVING porcentaje_inasistencia > 15.00
ORDER BY porcentaje_inasistencia DESC;

In [7]:
pipeline_c = [
    {
        "$group": {
            "_id": "$centro.nombre",
            "total_citas": { "$sum": 1 },
            "no_asistio": {
                "$sum": {
                    "$cond": [{ "$eq": ["$estado", "No Asistió"] }, 1, 0]
                }
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "centro": "$_id",
            "total_citas": 1,
            "no_asistio": 1,
            "porcentaje_inasistencia": {
                "$round": [
                    { "$multiply": [{ "$divide": ["$no_asistio", "$total_citas"] }, 100] },
                    2
                ]
            }
        }
    },
    {
        "$match": {
            "porcentaje_inasistencia": { "$gt": 15.0 }
        }
    },
    {
        "$sort": { "porcentaje_inasistencia": -1 }
    }
]

res_c = list(db.citas.aggregate(pipeline_c))
pprint(res_c)

[{'centro': 'Centro Médico San Joaquín',
  'no_asistio': 7,
  'porcentaje_inasistencia': 22.58,
  'total_citas': 31},
 {'centro': 'Centro Médico Casa Central',
  'no_asistio': 11,
  'porcentaje_inasistencia': 18.97,
  'total_citas': 58}]


### 4.3.d Ranking de medicamentos por especialidad

**Requerimiento:**
Dada una especialidad (Pediatría), mostrar los 5 medicamentos más recetados en las atenciones de esa especialidad junto a la cantidad de recetas de cada uno.

**Supuesto de Desempate (Documentado para README):**
En caso de presentarse una misma cantidad de recetas entre dos o más fármacos, se aplica como criterio de desempate el orden alfabético ascendente según el nombre del medicamento.

**Parámetros de prueba:**
* **Especialidad:** `'Pediatría'`

**Explicación de la solución MQL:**
1. `$match`: Filtra las citas correspondientes a la especialidad `'Pediatría'` que tengan el subarreglo `atencion.recetas`.
2. `$unwind`: Descompone el arreglo de recetas embebido, generando un documento individual por cada receta emitida.
3. `$group`: Agrupa por el atributo `atencion.recetas.medicamento` y totaliza las ocurrencias acumulando `$sum: 1`.
4. `$project`: Modela la presentación asignando el identificador al campo `medicamento` y renombrando la métrica como `cantidad_recetas`.
5. `$sort`: Aplica ordenamiento primario descendente (`-1`) por `cantidad_recetas` y secundario ascendente (`1`) por `medicamento` (criterio de desempate).
6. `$limit`: Limita la respuesta a los primeros 5 documentos para constituir el ranking.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT r.medicamento,
       COUNT(*) AS cantidad_recetas
FROM receta r
JOIN atencion a ON r.id_atencion = a.id_atencion
JOIN cita_medica c ON a.id_cita = c.id_cita
WHERE c.nombre_especialidad = 'Pediatría'
GROUP BY r.medicamento
ORDER BY cantidad_recetas DESC, r.medicamento ASC
LIMIT 5;

In [ ]:
pipeline_d = [
    {
        "$match": {
            "especialidad": "Pediatría",
            "atencion.recetas": { "$exists": True }
        }
    },
    {
        "$unwind": "$atencion.recetas"
    },
    {
        "$group": {
            "_id": "$atencion.recetas.medicamento",
            "cantidad_recetas": { "$sum": 1 }
        }
    },
    {
        "$project": {
            "_id": 0,
            "medicamento": "$_id",
            "cantidad_recetas": 1
        }
    },
    {
        "$sort": { "cantidad_recetas": -1, "medicamento": 1 }
    },
    {
        "$limit": 5
    }
]

res_d = list(db.citas.aggregate(pipeline_d))
pprint(res_d)

[{'cantidad_recetas': 8, 'medicamento': 'Paracetamol Gotas'},
 {'cantidad_recetas': 5, 'medicamento': 'Ibuprofeno Suspensión'},
 {'cantidad_recetas': 4, 'medicamento': 'Amoxicilina Suspensión'},
 {'cantidad_recetas': 2, 'medicamento': 'Cetirizina Jarabe'},
 {'cantidad_recetas': 2, 'medicamento': 'Mupirocina Crema'}]


### 4.3.e Pacientes multi-especialidad

**Requerimiento:**
Obtener los pacientes que han sido atendidos (citas en estado 'Atendida') en 3 o más especialidades distintas, mostrando su nombre, la cantidad de especialidades y el listado de ellas, ordenados de mayor a menor cantidad.

**Explicación de la solución MQL:**
1. `$match`: Selecciona exclusivamente los registros con estado `'Atendida'`.
2. `$group`: Agrupa por `paciente.rut`, conservando el nombre del paciente con `$first` y consolidando un conjunto de valores únicos para las disciplinas médicas usando el acumulador `$addToSet: "$especialidad"`.
3. `$project`: Modela los atributos de salida extrayendo el listado de especialidades y computando el tamaño del arreglo mediante el operador `$size`.
4. `$match`: Aplica la restricción de cardinalidad filtrando aquellos pacientes con `cantidad_especialidades` mayor o igual a 3 (`$gte: 3`).
5. `$sort`: Ordena de mayor a menor (`-1`) según la cantidad de especialidades y alfabéticamente (`1`) por el nombre del paciente.

**Consulta SQL equivalente (MySQL):**
```sql
SELECT p.nombre_completo AS paciente,
       COUNT(DISTINCT c.nombre_especialidad) AS cantidad_especialidades,
       GROUP_CONCAT(DISTINCT c.nombre_especialidad ORDER BY c.nombre_especialidad SEPARATOR ', ') AS listado_especialidades
FROM cita_medica c
JOIN paciente p ON c.rut_paciente = p.rut_paciente
WHERE c.estado = 'Atendida'
GROUP BY p.rut_paciente, p.nombre_completo
HAVING COUNT(DISTINCT c.nombre_especialidad) >= 3
ORDER BY cantidad_especialidades DESC;

In [9]:
pipeline_e = [
    {
        "$match": {
            "estado": "Atendida"
        }
    },
    {
        "$group": {
            "_id": "$paciente.rut",
            "nombre_paciente": { "$first": "$paciente.nombre" },
            "especialidades": { "$addToSet": "$especialidad" }
        }
    },
    {
        "$project": {
            "_id": 0,
            "paciente": "$nombre_paciente",
            "listado_especialidades": "$especialidades",
            "cantidad_especialidades": { "$size": "$especialidades" }
        }
    },
    {
        "$match": {
            "cantidad_especialidades": { "$gte": 3 }
        }
    },
    {
        "$sort": { "cantidad_especialidades": -1, "paciente": 1 }
    }
]

res_e = list(db.citas.aggregate(pipeline_e))
pprint(res_e)

[{'cantidad_especialidades': 5,
  'listado_especialidades': ['Neurología',
                             'Dermatología',
                             'Oftalmología',
                             'Medicina General',
                             'Traumatología'],
  'paciente': 'Juan Silva'},
 {'cantidad_especialidades': 4,
  'listado_especialidades': ['Pediatría',
                             'Dermatología',
                             'Neurología',
                             'Oftalmología'],
  'paciente': 'Camila Soto'},
 {'cantidad_especialidades': 4,
  'listado_especialidades': ['Traumatología',
                             'Cardiología',
                             'Medicina General',
                             'Ginecología'],
  'paciente': 'Pedro Morales'}]
